# Fine-tuning do YOLOv8 — african-wildlife

Treina com a GPU do Kaggle e produz um único artefato: **`best.pt`**. É a única coisa que vai para o Render.

## Antes de executar — dois ajustes obrigatórios

No painel **Settings**, à direita:

1. **Accelerator** → `GPU T4 x2` (ou `GPU P100`).
2. **Internet** → **ON**. Vem desligado por padrão. Sem isso o `pip install` falha e o dataset não baixa.

Cota: ~30 h de GPU por semana, sessão de até 12 h. Este treino usa poucos minutos.

## 1. Confirmar que a GPU está ativa

Se `cuda = False`, o Accelerator não foi selecionado. Pare e corrija — na CPU do Kaggle o treino leva horas.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
print("gpu :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA")

assert torch.cuda.is_available(), "Ative a GPU em Settings > Accelerator antes de seguir."

## 2. Dependências

O Kaggle já traz o PyTorch com CUDA; só falta o ultralytics. O pandas é usado pelo relatório do `fine_tuning.py`.

In [ ]:
!pip install -q ultralytics==8.4.172 pandas==2.3.3

import ultralytics

ultralytics.checks()

## 3. Trazer o código do repositório

Em vez de duplicar a lógica de treino aqui, o notebook clona o repositório e usa o `fine_tuning.py` versionado. Uma fonte da verdade só: o que roda aqui é exatamente o que está commitado.

Trabalhamos em `/kaggle/working` porque **`/kaggle/input` é somente leitura** — o ultralytics precisa escrever cache de labels e os artefatos do treino.

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/leo-badell/Projeto-Avaliativo-Modulo-2.git"
BRANCH = "fine_tuning"
DESTINO = Path("/kaggle/working/projeto")

if not DESTINO.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO} {DESTINO}

os.chdir(DESTINO)
print("diretorio:", Path.cwd())
print(sorted(p.name for p in Path.cwd().iterdir() if p.is_file()))

## 4. Treinar

O `african-wildlife.yaml` é resolvido e baixado pelo próprio ultralytics — ~1050 imagens de treino, 4 classes.

`--publicar` copia o `best.pt` para a raiz do repositório clonado, que é onde a API e o Dockerfile o esperam.

Ajuste `--epocas` conforme o tempo disponível. 50 épocas numa T4 ficam na casa de 15–25 min; o `--paciencia 10` interrompe antes se a validação estagnar.

In [ ]:
!python fine_tuning.py \
    --dataset african-wildlife.yaml \
    --epocas 50 \
    --batch 16 \
    --imgsz 640 \
    --paciencia 10 \
    --publicar

## 5. Curvas e matriz de confusão

O ultralytics grava os gráficos junto dos pesos. Servem para a apresentação.

In [ ]:
from IPython.display import Image, display

SAIDA = Path("runs/fine_tuning/african_wildlife")

for grafico in ["results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"]:
    caminho = SAIDA / grafico
    if caminho.exists():
        print(f"--- {grafico} ---")
        display(Image(filename=str(caminho), width=900))

## 6. Comparação antes/depois

Esta é a célula que demonstra o fine-tuning. A mesma imagem passa pelo modelo base e pelo ajustado.

`buffalo` e `rhino` não existem no COCO: o modelo base não tem como encontrá-los, independentemente da confiança.

In [ ]:
import pandas as pd
from ultralytics import YOLO

base = YOLO("yolov8n.pt")
ajustado = YOLO("best.pt")

# Uma imagem de validação do próprio dataset.
amostra = sorted(Path("/kaggle/working/datasets/african-wildlife/images/val").glob("*.jpg"))[0]
print("imagem:", amostra.name)


def deteccoes(modelo, rotulo):
    resultado = modelo.predict(source=str(amostra), conf=0.40, imgsz=640, verbose=False)[0]
    return pd.DataFrame(
        [
            {
                "modelo": rotulo,
                "classe": str(resultado.names[int(b.cls[0])]).lower(),
                "confianca": round(float(b.conf[0]) * 100, 2),
            }
            for b in resultado.boxes
        ]
    )


comparacao = pd.concat([deteccoes(base, "yolov8n"), deteccoes(ajustado, "best")], ignore_index=True)
display(comparacao)

# pivot deixa o contraste explícito: uma coluna por modelo, NaN onde não detectou.
display(
    comparacao.pivot_table(index="classe", columns="modelo", values="confianca", aggfunc="max")
)

## 7. Baixar o `best.pt`

A célula copia o peso para a raiz de `/kaggle/working`, de onde o painel **Output** (direita) permite baixá-lo.

Depois: coloque o arquivo na raiz do repositório local, commite e faça push. O Render reconstrói sozinho.

In [ ]:
import shutil

destino = Path("/kaggle/working/best.pt")
shutil.copy2("best.pt", destino)

print(f"{destino}  ({destino.stat().st_size / 1_048_576:.1f} MB)")
print("Baixe pelo painel Output, à direita.")